# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Global parameters
L_dom = 40.0                 # periodicno obmocje [-20, 20]
N = 512
x = np.linspace(-L_dom / 2, L_dom / 2, N, endpoint=False)
dx = x[1] - x[0]
k = 2 * np.pi * np.fft.fftfreq(N, d=dx)
dt = 1e-4
t_sol = 2.0                  # Naloga 1: cas za polno KdV
t_trk = 1.4                  # Naloga 2: cas preko trka

# Funkcije

In [ ]:
# Simbolno: disperzijska relacija in preverba solitona
xs, ts, ks, A, om = sp.symbols("x t k A omega", real=True)
u_val = A * sp.exp(sp.I * (ks * xs - om * ts))
disp = sp.solve(sp.simplify((sp.diff(u_val, ts) + sp.diff(u_val, xs, 3)) / u_val), om)[0]

c, kap = sp.symbols("c kappa", positive=True)
u_sol = 2 * kap**2 * sp.sech(kap * (xs - 4 * kap**2 * ts))**2
kdv = lambda u: sp.diff(u, ts) + 6 * u * sp.diff(u, xs) + sp.diff(u, xs, 3)

In [ ]:
def soliton(x, kap, x0, t=0.0):
    # 2 kappa^2 sech^2(kappa (x - x0 - 4 kappa^2 t)), amplituda 2 kappa^2, hitrost 4 kappa^2
    return 2 * kap**2 / np.cosh(kap * (x - x0 - 4 * kap**2 * t))**2

dealias = np.abs(k) < (2 / 3) * np.abs(k).max()

def kdv_ifrk4(u0, t_end, dt=dt, n_shrani=100, disperzija=True, nelinearnost=True):
    # psevdospektralno: u_t = -u_xxx - 3 (u^2)_x ; linearni del eksaktno z integracijskim faktorjem
    lin = 1j * k**3 if disperzija else 0 * k
    E = np.exp(lin * dt / 2)
    def nel(uh):
        if not nelinearnost:
            return 0 * uh
        u = np.real(np.fft.ifft(uh))
        return -3j * k * np.fft.fft(u**2) * dealias
    uh = np.fft.fft(u0)
    n_kor = int(round(t_end / dt))
    t_s, u_s = [0.0], [u0.copy()]
    for n in range(1, n_kor + 1):
        a = dt * nel(uh)
        b = dt * nel(E * (uh + a / 2))
        c_ = dt * nel(E * uh + b / 2)
        d = dt * nel(E**2 * uh + E * c_)
        uh = E**2 * uh + (E**2 * a + 2 * E * (b + c_) + d) / 6
        if n % n_shrani == 0:
            t_s.append(n * dt)
            u_s.append(np.real(np.fft.ifft(uh)))
    return np.array(t_s), np.array(u_s)

def vrh(x, u):
    # polozaj vrha s parabolicno interpolacijo
    i = np.argmax(u)
    ym, y0, yp = u[i - 1], u[i], u[(i + 1) % len(u)]
    return x[i] + 0.5 * dx * (ym - yp) / (ym - 2 * y0 + yp), y0

def karakteristike(u0_f, xi, t):
    # samo nelinearnost: u = u0(xi), x = xi + 6 u0(xi) t (do loma vala)
    return xi + 6 * u0_f(xi) * t, u0_f(xi)

# Testno območje

In [ ]:
# Pripravljalna naloga 1: disperzijska relacija, fazna in grupna hitrost
display(sp.Eq(om, disp))
print("v_f = omega/k =", sp.simplify(disp / ks), ",  v_g = d omega/dk =", sp.diff(disp, ks))

# Pripravljalna naloga 2: u = 2 kappa^2 sech^2(kappa(x - 4 kappa^2 t)) resi KdV; kappa = 1 -> 2 sech^2(x - 4t)
print("KdV ostanek (splosen kappa):", sp.simplify(kdv(u_sol).rewrite(sp.exp)))
print("amplituda A = 2 kappa^2, hitrost c = 4 kappa^2  ->  c = 2A")

# Naloga 1

In [ ]:
u0 = soliton(x, 1.0, 0.0)
casi = [0.0, 0.5, 1.0, 2.0]

fig = plt.figure(figsize=(16, 4.5))
# (a) samo disperzija: u_t + u_xxx = 0
t_d, u_d = kdv_ifrk4(u0, 2.0, dt=1e-3, n_shrani=1, nelinearnost=False)
ax = fig.add_subplot(131)
for tc in casi:
    ax.plot(x, u_d[int(round(tc / 1e-3))], label=f"$t = {tc}$")
ax.set_title(r"samo disperzija: $u_t + u_{xxx} = 0$")
ax.set_xlabel("$x$", fontsize=14)
ax.set_ylabel("$u$", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)

# (b) samo nelinearnost: u_t + 6 u u_x = 0 (karakteristike)
u0_f = lambda xi: soliton(xi, 1.0, 0.0)
xi = np.linspace(-20, 20, 4001)
t_lom = -1 / (6 * np.min(np.gradient(u0_f(xi), xi)))
ax = fig.add_subplot(132)
for tc in [0.0, 0.05, t_lom]:
    xx, uu = karakteristike(u0_f, xi, tc)
    ax.plot(xx, uu, label=f"$t = {tc:.3f}$")
ax.set_xlim(-5, 8)
ax.set_title(rf"samo nelinearnost: $u_t + 6uu_x = 0$, $t_{{lom}} = {t_lom:.3f}$")
ax.set_xlabel("$x$", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)

# (c) polna KdV
t_k, u_k = kdv_ifrk4(u0, t_sol, n_shrani=100)
ax = fig.add_subplot(133)
for tc in casi:
    i = int(round(tc / (100 * dt)))
    ax.plot(x, u_k[i], label=f"$t = {tc}$")
    ax.plot(x, soliton(x, 1.0, 0.0, tc), ":", color="black")
ax.set_title("polna KdV (črtkano: analitično)")
ax.set_xlabel("$x$", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

poz = np.array([vrh(x, u)[0] for u in u_k])
hitrost = np.polyfit(t_k, poz, 1)[0]
print(f"izmerjena hitrost solitona c = {hitrost:.4f}  (teorija 4),  max |u - u_an| pri t = {t_sol}: "
      f"{np.max(np.abs(u_k[-1] - soliton(x, 1.0, 0.0, t_sol))):.1e}")

# Naloga 2

In [ ]:
kap1, x1 = 2.0, -8.0      # vecji, hitrejsi (A = 8, c = 16)
kap2, x2 = 1.0, 2.0       # manjsi, pocasnejsi (A = 2, c = 4)
u0_2 = soliton(x, kap1, x1) + soliton(x, kap2, x2)
t_2, u_2 = kdv_ifrk4(u0_2, t_trk, n_shrani=20)

fig = plt.figure(figsize=(14, 5))
ax = fig.add_subplot(121)
im = ax.imshow(u_2, origin="lower", aspect="auto", extent=[x[0], x[-1], 0, t_trk], cmap="viridis")
ax.plot(x1 + 4 * kap1**2 * t_2, t_2, "w--", linewidth=0.8)
ax.plot(x2 + 4 * kap2**2 * t_2, t_2, "w:", linewidth=0.8)
ax.set_xlim(x[0], x[-1])
ax.set_xlabel("$x$", fontsize=14)
ax.set_ylabel("$t$", fontsize=14)
fig.colorbar(im, ax=ax, label="$u$")

ax = fig.add_subplot(122)
for i, tc in enumerate([0.0, 0.5, 0.75, t_trk]):
    j = int(round(tc / (20 * dt)))
    ax.plot(x, u_2[j] + 3 * i, label=f"$t = {tc}$")
ax.set_xlabel("$x$", fontsize=14)
ax.set_ylabel("$u$ (zamaknjeno)", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Amplitudi in fazna premika po trku
u_kon = u_2[-1]
lev, des = x < 11, x >= 11            # po trku: manjsi levo, vecji desno
p2, A2 = vrh(x[lev], u_kon[lev])
p1, A1 = vrh(x[des], u_kon[des])
d1_num, d2_num = p1 - (x1 + 4 * kap1**2 * t_trk), p2 - (x2 + 4 * kap2**2 * t_trk)
d1_an = np.log((kap1 + kap2) / (kap1 - kap2)) / kap1
d2_an = -np.log((kap1 + kap2) / (kap1 - kap2)) / kap2
print(f"vecji:  A = {A1:.3f} (zacetno 8),  premik = {d1_num:+.3f}  (teorija {d1_an:+.3f})")
print(f"manjsi: A = {A2:.3f} (zacetno 2),  premik = {d2_num:+.3f}  (teorija {d2_an:+.3f})")

# Naloga 3

In [ ]:
def invariante(u):
    ux = np.real(np.fft.ifft(1j * k * np.fft.fft(u)))
    return np.sum(u) * dx, np.sum(u**2) * dx, np.sum(u**3 - 0.5 * ux**2) * dx

I = np.array([invariante(u) for u in u_2])
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111)
for j, ime in enumerate([r"$I_1 = \int u\,dx$", r"$I_2 = \int u^2 dx$", r"$I_3 = \int (u^3 - u_x^2/2)\,dx$"]):
    ax.plot(t_2, I[:, j] / I[0, j] - 1, label=ime)
    print(f"{ime:35s}: zacetno {I[0, j]:10.5f},  max relativna sprememba {np.max(np.abs(I[:, j] / I[0, j] - 1)):.1e}")
ax.set_xlabel("$t$", fontsize=14)
ax.set_ylabel("relativna sprememba", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.show()